In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# RBI Policy Dates
rbi_dates = [
    '2024-02-08',
    '2024-04-05',
    '2024-06-07',
    '2024-08-08',
    '2024-10-09',
    '2024-12-06',
    '2025-02-07',
    '2025-04-09',
]

# Pull Nifty data
nifty = yf.Ticker("^NSEI")
hist = nifty.history(start="2024-01-01", end="2025-06-01")
hist['returns'] = hist['Close'].pct_change()
hist['HV10'] = hist['returns'].rolling(10).std() * np.sqrt(252) * 100

print("Data loaded:", hist.shape)
print(hist.tail(3))

In [ ]:
results = []

for date in rbi_dates:
    rbi_date = pd.Timestamp(date).tz_localize('Asia/Kolkata')

    # Get HV 5 days before and 5 days after
    start = rbi_date - pd.Timedelta(days=10)
    end = rbi_date + pd.Timedelta(days=10)

    window = hist[(hist.index >= start) & (hist.index <= end)]['HV10'].dropna()

    if len(window) >= 4:
        # Find RBI date position
        before_dates = hist[(hist.index >= start) & (hist.index < rbi_date)]['HV10'].dropna()
        after_dates = hist[(hist.index > rbi_date) & (hist.index <= end)]['HV10'].dropna()

        if len(before_dates) >= 2 and len(after_dates) >= 2:
            hv_before = before_dates.iloc[-1]  # HV day before RBI
            hv_after = after_dates.iloc[1]     # HV 2 days after RBI
            hv_change = hv_after - hv_before
            hv_change_pct = (hv_change / hv_before) * 100

            results.append({
                'rbi_date': date,
                'hv_before': round(hv_before, 2),
                'hv_after': round(hv_after, 2),
                'hv_change': round(hv_change, 2),
                'hv_change_pct': round(hv_change_pct, 2),
                'iv_crushed': hv_after < hv_before
            })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), facecolor='#0a0a0a')

dates = results_df['rbi_date']
x = range(len(dates))
width = 0.35

# Chart 1 - HV Before vs After
ax1.set_facecolor('#0a0a0a')
ax1.bar([i - width/2 for i in x], results_df['hv_before'],
        width, label='HV Before RBI', color='#FF6B6B', alpha=0.9)
ax1.bar([i + width/2 for i in x], results_df['hv_after'],
        width, label='HV After RBI', color='#4ECDC4', alpha=0.9)
ax1.set_xticks(list(x))
ax1.set_xticklabels(dates, rotation=45, fontsize=9, color='white')
ax1.set_title('HV Before vs After RBI Dates', color='white', fontsize=12, pad=10)
ax1.set_ylabel('Historical Volatility %', color='white')
ax1.tick_params(colors='white')
ax1.legend(facecolor='#1a1a1a', labelcolor='white')
ax1.grid(True, alpha=0.2, color='white')
ax1.spines['bottom'].set_color('#333')
ax1.spines['left'].set_color('#333')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Chart 2 - HV Change %
ax2.set_facecolor('#0a0a0a')
bar_colors = ['#2ECC71' if v < 0 else '#E74C3C'
              for v in results_df['hv_change_pct']]
bars = ax2.bar(list(x), results_df['hv_change_pct'],
               color=bar_colors, alpha=0.9, width=0.5)
ax2.axhline(y=0, color='white', linewidth=1, alpha=0.5)
ax2.set_xticks(list(x))
ax2.set_xticklabels(dates, rotation=45, fontsize=9, color='white')
ax2.set_title('HV Change % After RBI  |  Green = IV Crush  |  Red = IV Spike',
              color='white', fontsize=12, pad=10)
ax2.set_ylabel('HV Change %', color='white')
ax2.tick_params(colors='white')
ax2.grid(True, alpha=0.2, color='white')
ax2.spines['bottom'].set_color('#333')
ax2.spines['left'].set_color('#333')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add value labels on bars
for i, v in enumerate(results_df['hv_change_pct']):
    ax2.text(i, v + (1 if v > 0 else -2), f'{v}%',
             ha='center', va='bottom', color='white', fontsize=8)

plt.suptitle('Nifty Event Calendar Backtest — RBI 2024-2025',
             fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.savefig('event_backtest.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()

# Summary
crushed = results_df['iv_crushed'].sum()
total = len(results_df)
print(f"\nIV Crush rate: {crushed}/{total} events ({round(crushed/total*100)}%)")
print(f"Average HV change: {round(results_df['hv_change_pct'].mean(), 2)}%")
print(f"Avg crush magnitude: {round(results_df[results_df['iv_crushed']]['hv_change_pct'].mean(), 2)}%")
print(f"Avg spike magnitude: {round(results_df[~results_df['iv_crushed']]['hv_change_pct'].mean(), 2)}%")

## Nifty Event Calendar Backtest — RBI 2024-2025

### Strategy Tested
Sell options just before RBI announcement, buy back after.
Profits when IV crushes. Loses when IV spikes.

### Results
- IV Crush rate: 4/8 events (50%)
- Average HV change after RBI: -3.14%
- Average crush magnitude: -18.5%
- Average spike magnitude: +10.7%

### Key Finding
IV crush happens more often than spikes but spike magnitude
can be severe — Feb 2025 rate cut and Apr 2025 continuation
both caused significant post-event volatility expansion.

### Trading Conclusion
Blind short volatility before every RBI = risky.
Selective short volatility only when outcome highly expected = better edge.
Always size positions small around event risk
